In [ ]:
import os, glob, time, gzip, shutil
import numpy as np
import pandas as pd

# ---------- الإعدادات ----------
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
BIG_FILE = os.path.join(WORK, "big_voice.csv")
REPEAT = 5000          # 3168 سطر × 1000 ≈ 3.2 مليون سطر (~1GB). ارفعها إلى 5000+ للوصول إلى 5GB+
CHUNK_SIZE = 500_000   # عدد الأسطر في كل chunk
LABEL = "label"

def size_mb(p): return os.path.getsize(p) / 1024**2

def timed(f, *a, **k):
    t0 = time.perf_counter(); out = f(*a, **k)
    return out, time.perf_counter() - t0

In [ ]:
cands = glob.glob("/kaggle/input/datasets/primaryobjects/voicegender/voice.csv", recursive=True) + glob.glob("/kaggle/input/datasets/primaryobjects/voicegender/voice.csv")
cands = [c for c in cands if "big_voice" not in c]
print("Source file:", cands[:1])
base = pd.read_csv(cands[0])
print(base.shape)
base.head()

In [ ]:
if not os.path.exists(BIG_FILE):
    for i in range(REPEAT):
        base.to_csv(BIG_FILE, mode="w" if i == 0 else "a", header=(i == 0), index=False)
print(f"{BIG_FILE}: {size_mb(BIG_FILE):.0f} MB, rows = {len(base)*REPEAT:,}")
NUM_COLS = [c for c in base.columns if c != LABEL]

## 1) القراءة الكاملة (baseline)
تحمّل كل الملف في الذاكرة مرة واحدة. تنجح فقط إذا اتسعت الذاكرة، وإلا يظهر `MemoryError`.

In [ ]:
def method_full_load(path):
    df = pd.read_csv(path)
    ram = df.memory_usage(deep=True).sum() / 1024**2
    return df.groupby(LABEL).mean(numeric_only=True), ram

rows = []
try:
    (res_full, ram), t_full = timed(method_full_load, BIG_FILE)
    rows.append(["read_csv (full load)", t_full, size_mb(BIG_FILE), f"{ram:.0f} MB in RAM"])
    print(f"Full load: {t_full:.1f}s | RAM ~ {ram:.0f} MB")
except MemoryError:
    res_full = None
    rows.append(["read_csv (full load)", float("nan"), size_mb(BIG_FILE), "MemoryError"])
    print("MemoryError!")

## 2) `pandas.read_csv(chunksize=...)`
نقرأ الملف على دفعات. لكل دفعة نحسب **المجموع** و**العدد** لكل label، وفي النهاية: المتوسط = مجموع المجاميع ÷ مجموع الأعداد.  
في أي لحظة توجد دفعة واحدة فقط في الذاكرة.

In [ ]:
def method_chunks(path, compression="infer", dtype=None):
    sums, counts, peak = None, None, 0
    for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE, compression=compression, dtype=dtype):
        peak = max(peak, chunk.memory_usage(deep=True).sum() / 1024**2)
        g = chunk.groupby(LABEL)
        s, c = g.sum(numeric_only=True), g.size()
        sums = s if sums is None else sums.add(s, fill_value=0)
        counts = c if counts is None else counts.add(c, fill_value=0)
    return sums.div(counts, axis=0), peak

(res_chunk, peak), t = timed(method_chunks, BIG_FILE)
rows.append(["pandas chunksize", t, size_mb(BIG_FILE), f"{peak:.0f} MB peak/chunk"])
print(f"Chunks: {t:.1f}s | peak per chunk ~ {peak:.0f} MB")
res_chunk

### تحسين: استخدام `float32` بدل `float64` لتقليل الذاكرة إلى النصف تقريباً

In [ ]:
dtypes = {c: "float32" for c in NUM_COLS}
(res_f32, peak32), t = timed(method_chunks, BIG_FILE, "infer", dtypes)
rows.append(["chunksize + float32", t, size_mb(BIG_FILE), f"{peak32:.0f} MB peak/chunk"])
print(f"Chunks float32: {t:.1f}s | peak per chunk ~ {peak32:.0f} MB")

## 3) Dask
يقسم الملف إلى partitions (64MB لكل واحد) ويعالجها **بالتوازي** على كل أنوية المعالج. التنفيذ كسول: لا يبدأ الحساب إلا عند `.compute()`.

In [ ]:
import dask.dataframe as dd

def method_dask(path):
    ddf = dd.read_csv(path, blocksize="64MB")
    return ddf.groupby(LABEL).mean().compute(), ddf.npartitions

(res_dask, parts), t = timed(method_dask, BIG_FILE)
rows.append(["Dask", t, size_mb(BIG_FILE), f"{parts} partitions"])
print(f"Dask: {t:.1f}s | partitions = {parts}")
res_dask

## 4) الضغط (Compression)
**gzip:** نضغط CSV إلى `.csv.gz` ثم نقرأه مباشرة مع `chunksize`.  
**Parquet:** صيغة عمودية مضغوطة، تسمح بقراءة أعمدة محددة فقط.

In [ ]:
GZ_FILE = BIG_FILE + ".gz"

def compress_gzip(src, dst):
    with open(src, "rb") as fi, gzip.open(dst, "wb", compresslevel=6) as fo:
        shutil.copyfileobj(fi, fo)

_, t_comp = timed(compress_gzip, BIG_FILE, GZ_FILE)
(res_gz, _), t = timed(method_chunks, GZ_FILE, "gzip")
ratio = size_mb(BIG_FILE) / size_mb(GZ_FILE)
rows.append(["gzip CSV + chunks", t, size_mb(GZ_FILE), f"compress {t_comp:.1f}s, ratio {ratio:.1f}x"])
print(f"gzip: size {size_mb(GZ_FILE):.0f} MB (ratio {ratio:.1f}x) | read {t:.1f}s | compress {t_comp:.1f}s")

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

PQ_FILE = os.path.join(WORK, "big_voice.parquet")

def to_parquet(src, dst):
    writer = None
    for chunk in pd.read_csv(src, chunksize=CHUNK_SIZE):
        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(dst, table.schema, compression="snappy")
        writer.write_table(table)
    writer.close()

_, t_conv = timed(to_parquet, BIG_FILE, PQ_FILE)

use_cols = NUM_COLS[:5] + [LABEL]
t0 = time.perf_counter()
res_pq = pd.read_parquet(PQ_FILE, columns=use_cols).groupby(LABEL).mean(numeric_only=True)
t = time.perf_counter() - t0
rows.append(["Parquet (6 columns only)", t, size_mb(PQ_FILE), f"conversion {t_conv:.1f}s"])
print(f"Parquet: size {size_mb(PQ_FILE):.0f} MB | read {t:.1f}s | conversion {t_conv:.1f}s")

## 5) التحقق من أن كل الطرق تعطي نفس النتيجة

In [ ]:
ref = res_chunk.sort_index()
for name, r in [("full", res_full), ("dask", res_dask), ("gzip", res_gz)]:
    if r is not None:
        ok = np.allclose(ref.values, r.sort_index()[ref.columns].values, rtol=1e-4)
        print(f"chunks vs {name}: {'OK' if ok else 'DIFFERENT'}")

## 6) جدول المقارنة (الوقت والتخزين)

In [ ]:
table = pd.DataFrame(rows, columns=["Method", "Time (s)", "File size (MB)", "Notes"])
table["Time (s)"] = table["Time (s)"].round(2)
table["File size (MB)"] = table["File size (MB)"].round(1)
table.to_csv(os.path.join(WORK, "comparison_results.csv"), index=False)
table

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
table.plot.barh(x="Method", y="Time (s)", ax=ax[0], legend=False, title="Time (seconds)")
table.plot.barh(x="Method", y="File size (MB)", ax=ax[1], legend=False, title="File size on disk (MB)", color="tab:orange")
plt.tight_layout(); plt.show()

In [ ]:
import json, platform, os

results = {
    "dataset": {
        "source_rows": int(len(base)),
        "repeat": REPEAT,
        "total_rows": int(len(base) * REPEAT),
        "csv_size_mb": round(size_mb(BIG_FILE), 1),
        "chunk_size": CHUNK_SIZE,
    },
    "environment": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "cpu_count": os.cpu_count(),
    },
    # to_json يحوّل NaN إلى null تلقائياً
    "results": json.loads(table.to_json(orient="records")),
}

path = os.path.join(WORK, "results.json")
with open(path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved:", path)
print(json.dumps(results, indent=2))